# Logistic Regression (Loan Approval Prediction)
## Assignment Submission

This notebook covers:
- Data Loading & Understanding
- Data Cleaning
- Exploratory Data Analysis (EDA)
- Outlier Detection & Treatment
- Feature Selection & Data Splitting
- Feature Scaling
- Logistic Regression Model Building
- Confusion Matrix
- ROC Curve & AUC
- Regression Equation
- Interpretation of Coefficients

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score


# Q1. Data Loading & Understanding

In [ ]:
df = pd.read_excel('loan_approval.xlsx')
display(df.head())
print('Shape:', df.shape)
print(df.dtypes)

# Q2. Data Cleaning

In [ ]:
print(df.isnull().sum())

# Handle missing values if any
for col in df.select_dtypes(include='number').columns:
    df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(exclude='number').columns:
    df[col] = df[col].fillna(df[col].mode()[0])

categorical_cols = df.select_dtypes(exclude='number').columns.tolist()
print('Categorical Columns:', categorical_cols)

# Q3. Exploratory Data Analysis (EDA)

In [ ]:
df['loan_approved'].value_counts().plot(kind='bar')
plt.title('Target Variable Distribution')
plt.show()

In [ ]:
credit_col = 'credit_score'
df.groupby('loan_approved')[credit_col].mean().plot(kind='bar')
plt.title('Credit Score vs Loan Approval')
plt.show()

In [ ]:
emp_col = 'years_employed'
df.groupby('loan_approved')[emp_col].mean().plot(kind='bar')
plt.title('Years Employed vs Loan Approval')
plt.show()

# Q4. Outlier Detection and Treatment

In [ ]:
num_cols = df.select_dtypes(include='number').columns.tolist()
num_cols.remove('loan_approved')

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5*IQR
    upper = Q3 + 1.5*IQR
    df[col] = np.clip(df[col], lower, upper)

print('Outliers treated using IQR capping.')

# Q5. Convert Target Variable and Drop Unnecessary Columns

In [ ]:
df['loan_approved'] = df['loan_approved'].astype(int)
df = df.drop(columns=['name','city'], errors='ignore')
df.head()

# Q6. Feature Selection and Data Splitting

In [ ]:
X = df.drop('loan_approved', axis=1)
y = df['loan_approved']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Q7. Feature Scaling

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Q8. Logistic Regression Model Building

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:,1]
print('Intercept:', model.intercept_[0])
print('Coefficients:', dict(zip(X.columns, model.coef_[0])))

# Regression Equation

In [ ]:
equation = f'log(p/(1-p)) = {model.intercept_[0]:.4f}'
for col, coef in zip(X.columns, model.coef_[0]):
    equation += f' + ({coef:.4f}*{col})'
print(equation)

# Q9. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.show()
print(cm)

# Q10. ROC Curve & AUC

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)
plt.plot(fpr, tpr)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title(f'ROC Curve (AUC={auc:.4f})')
plt.show()
print('AUC Score:', auc)

# Interpretation of Coefficients
- Positive coefficient: increases probability of loan approval.
- Negative coefficient: decreases probability of loan approval.
- Larger magnitude indicates stronger influence on prediction.